In [11]:
import os
import duckdb
import boto3
from dotenv import load_dotenv
import pandas as pd

Configuração do Pandas

In [12]:
pd.set_option('display.max_columns', None)

Carregando as variáveis do arquivo .env

In [2]:
load_dotenv()

PROFILE = os.getenv("AWS_PROFILE")
REGION = os.getenv("AWS_REGION")
PREFIX = os.getenv("PROJECT_BUCKET_PREFIX")
BRONZE_BUCKET = f"{PREFIX}-bronze"

Definindo as conexões com o S3 e o DuckDB

In [4]:
session = boto3.Session(profile_name=PROFILE, region_name=REGION)
creds = session.get_credentials().get_frozen_credentials()

con = duckdb.connect(':memory:')
con.execute("INSTALL httpfs; LOAD httpfs; INSTALL aws; LOAD aws;")
con.execute(f"SET s3_region='{REGION}';")
con.execute(f"SET s3_access_key_id='{creds.access_key}';")
con.execute(f"SET s3_secret_access_key='{creds.secret_key}';")

Carregando arquivo do Bucket S3

In [5]:
parquet_path = f"s3://{BRONZE_BUCKET}/perfil_comparecimento_abstencao/2022/perfil_comparecimento_abstencao_2022_BRASIL.parquet"

#### Schema do Arquivo

In [9]:
con.execute(f"DESCRIBE SELECT * FROM '{parquet_path}'").df()


,column_name,column_type,null,key,default,extra
0,DT_GERACAO,VARCHAR,YES,None,None,None
1,HH_GERACAO,VARCHAR,YES,None,None,None
2,ANO_ELEICAO,VARCHAR,YES,None,None,None
3,NR_TURNO,VARCHAR,YES,None,None,None
4,SG_UF,VARCHAR,YES,None,None,None
5,CD_MUNICIPIO,VARCHAR,YES,None,None,None
6,NM_MUNICIPIO,VARCHAR,YES,None,None,None
7,NR_ZONA,VARCHAR,YES,None,None,None
8,CD_GENERO,VARCHAR,YES,None,None,None
9,DS_GENERO,VARCHAR,YES,None,None,None


#### Amostra dos Dados

In [13]:
con.execute(f"SELECT * FROM '{parquet_path}' LIMIT 10").df()

,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,NR_TURNO,SG_UF,CD_MUNICIPIO,NM_MUNICIPIO,NR_ZONA,CD_GENERO,DS_GENERO,CD_ESTADO_CIVIL,DS_ESTADO_CIVIL,CD_FAIXA_ETARIA,DS_FAIXA_ETARIA,CD_GRAU_ESCOLARIDADE,DS_GRAU_ESCOLARIDADE,CD_COR_RACA,DS_COR_RACA,CD_QUILOMBOLA,DS_QUILOMBOLA,CD_INTERPRETE_LIBRAS,DS_INTERPRETE_LIBRAS,CD_IDENTIDADE_GENERO,DS_IDENTIDADE_GENERO,CD_IDIOMA_INDIGENA,DS_IDIOMA_INDIGENA,CD_GRUPO_INDIGENA,DS_GRUPO_INDIGENA,QT_APTOS,QT_COMPARECIMENTO,QT_ABSTENCAO,QT_COMPARECIMENTO_DEFICIENCIA,QT_ABSTENCAO_DEFICIENCIA,QT_COMPARECIMENTO_TTE,QT_ABSTENCAO_TTE,QT_COMPAREC_FACULTATIVO,QT_ABST_FACULTATIVO,QT_COMPAREC_OBRIGATORIO,QT_ABST_OBRIGATORIO,QT_COMPAREC_DEFIC_FACULTATIVO,QT_ABST_DEFIC_FACULTATIVO,QT_COMPAREC_DEFIC_OBRIGATORIO,QT_ABST_DEFIC_OBRIGATORIO
0,29/04/2025,22:31:40,2022,1,PR,74004,GODOY MOREIRA,132,2,MASCULINO,1,SOLTEIRO,1600,16 anos,5,ENSINO MÉDIO INCOMPLETO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,9,8,1,0,0,0,0,8,1,0,0,0,0,0,0
1,29/04/2025,22:31:40,2022,1,PR,74004,GODOY MOREIRA,132,2,MASCULINO,1,SOLTEIRO,1700,17 anos,3,ENSINO FUNDAMENTAL INCOMPLETO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0
2,29/04/2025,22:31:40,2022,1,PR,74004,GODOY MOREIRA,132,2,MASCULINO,1,SOLTEIRO,1800,18 anos,3,ENSINO FUNDAMENTAL INCOMPLETO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,3,2,1,0,0,0,0,0,0,2,1,0,0,0,0
3,29/04/2025,22:31:40,2022,1,PR,74004,GODOY MOREIRA,132,2,MASCULINO,1,SOLTEIRO,1900,19 anos,5,ENSINO MÉDIO INCOMPLETO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,14,11,3,0,0,0,0,0,0,11,3,0,0,0,0
4,29/04/2025,22:31:40,2022,1,PR,74004,GODOY MOREIRA,132,2,MASCULINO,1,SOLTEIRO,1900,19 anos,6,ENSINO MÉDIO COMPLETO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,4,3,1,0,0,0,0,0,0,3,1,0,0,0,0
5,29/04/2025,22:31:40,2022,1,PR,74004,GODOY MOREIRA,132,2,MASCULINO,1,SOLTEIRO,2124,21 a 24 anos,5,ENSINO MÉDIO INCOMPLETO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,38,29,9,0,0,0,0,0,0,29,9,0,0,0,0
6,29/04/2025,22:31:40,2022,1,PR,74004,GODOY MOREIRA,132,2,MASCULINO,1,SOLTEIRO,2124,21 a 24 anos,6,ENSINO MÉDIO COMPLETO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,22,12,10,0,0,0,0,0,0,12,10,0,0,0,0
7,29/04/2025,22:31:40,2022,1,PR,74004,GODOY MOREIRA,132,2,MASCULINO,1,SOLTEIRO,2529,25 a 29 anos,4,ENSINO FUNDAMENTAL COMPLETO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,4,2,2,0,0,0,0,0,0,2,2,0,0,0,0
8,29/04/2025,22:31:40,2022,1,PR,74004,GODOY MOREIRA,132,2,MASCULINO,1,SOLTEIRO,2529,25 a 29 anos,5,ENSINO MÉDIO INCOMPLETO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,11,5,6,0,0,0,0,0,0,5,6,0,0,0,0
9,29/04/2025,22:31:40,2022,1,PR,74004,GODOY MOREIRA,132,2,MASCULINO,1,SOLTEIRO,2529,25 a 29 anos,6,ENSINO MÉDIO COMPLETO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,48,37,11,0,0,0,0,0,0,37,11,0,0,0,0
